In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

In [ ]:
sales_df = pd.read_csv('Chocolate Sales.csv')

In [ ]:
sales_df.head(8)

In [ ]:
sales_df.tail(8)

In [ ]:
sales_df.shape

In [ ]:
sales_df.columns

In [ ]:
sales_df.info()

In [ ]:
sales_df.describe()

In [ ]:
sales_df['Amount'] = (
    sales_df['Amount']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)


In [ ]:
sales_df['Date'] = pd.to_datetime(sales_df['Date'], format='%d/%m/%Y')

In [ ]:
sales_df.info()

In [ ]:
sales_df.groupby('Country')['Amount'].sum().sort_values(ascending=False)

In [ ]:
sales_df.groupby('Product')['Boxes Shipped'].sum().sort_values(ascending=False)

In [ ]:
sales_df.groupby('Sales Person')['Amount'].sum().sort_values(ascending=False)

In [ ]:
sales_df['Month'] = sales_df['Date'].dt.to_period('M')
sales_df.groupby('Month')['Amount'].sum()


In [ ]:

country_sales_summary = sales_df.groupby('Country')['Amount'].sum().sort_values()

country_sales_summary.plot(kind='bar')
plt.title('Total Sales by Country')
plt.xlabel('Total Sales Amount')
plt.ylabel('Country')
plt.show()

In [ ]:
product_sales_summary = sales_df.groupby('Product')['Amount'].sum().sort_values(ascending=False)

product_sales_summary.plot(kind='bar')
plt.title('Total Sales by Product')
plt.ylabel('Sales Amount')
plt.xticks(rotation=90)
plt.show()

In [ ]:
plt.scatter(sales_df['Boxes Shipped'], sales_df['Amount'])
plt.xlabel('Boxes Shipped')
plt.ylabel('Sales Amount')
plt.title('Boxes Shipped vs Amount')
plt.show()

In [ ]:
monthly_sales_summary = sales_df.groupby(sales_df['Date'].dt.to_period('M'))['Amount'].sum()

monthly_sales_summary.plot(kind='line', marker='o')
plt.title('Monthly Sales Trend')
plt.xlabel('Month')
plt.ylabel('Sales Amount')
plt.show()

In [ ]:
sales_df['Year'] = sales_df['Date'].dt.year
sales_df['Month'] = sales_df['Date'].dt.month
sales_df['Day'] = sales_df['Date'].dt.day

In [ ]:
median_amount_threshold = sales_df['Amount'].median()

sales_df['high_sales_flag'] = (sales_df['Amount'] > median_amount_threshold).astype(int)

In [ ]:
feature_data = sales_df[['Country', 'Product', 'Boxes Shipped', 'Date']]
target_data = sales_df['high_sales_flag']

In [ ]:
feature_data['Month'] = feature_data['Date'].dt.month
feature_data['Day'] = feature_data['Date'].dt.day
feature_data = feature_data.drop(columns=['Date'])

In [ ]:
features_train, features_test, labels_train, labels_test = train_test_split(
    feature_data, target_data, test_size=0.2, random_state=42, stratify=target_data
)

In [ ]:
categorical_features = ['Country', 'Product']
numeric_features = ['Boxes Shipped', 'Month', 'Day']

feature_preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', StandardScaler(), numeric_features)
    ]
)


In [ ]:
logistic_pipeline = make_pipeline(
    feature_preprocessor,
    LogisticRegression(max_iter=1000)
)

logistic_params = {
    'logisticregression__C': [0.01, 0.1, 1, 10]
}

logistic_search = GridSearchCV(logistic_pipeline, logistic_params, scoring='roc_auc', cv=5)
logistic_search.fit(features_train, labels_train)


In [ ]:
knn_pipeline = make_pipeline(
    feature_preprocessor,
    KNeighborsClassifier()
)

knn_params = {
    'kneighborsclassifier__n_neighbors': [3,5,7,9],
    'kneighborsclassifier__weights': ['uniform', 'distance']
}

knn_search = GridSearchCV(knn_pipeline, knn_params, scoring='roc_auc', cv=5)
knn_search.fit(features_train, labels_train)


In [ ]:
svm_pipeline = make_pipeline(
    feature_preprocessor,
    SVC(probability=True)
)

svm_params = {
    'svc__C': [0.1, 1, 10],
    'svc__kernel': ['linear', 'rbf']
}

svm_search = GridSearchCV(svm_pipeline, svm_params, scoring='roc_auc', cv=5)
svm_search.fit(features_train, labels_train)


In [ ]:
decision_tree_pipeline = make_pipeline(
    feature_preprocessor,
    DecisionTreeClassifier(random_state=42)
)

decision_tree_params = {
    'decisiontreeclassifier__max_depth': [3,5,10,None],
    'decisiontreeclassifier__min_samples_split': [2,5,10]
}

decision_tree_search = GridSearchCV(decision_tree_pipeline, decision_tree_params, scoring='roc_auc', cv=5)
decision_tree_search.fit(features_train, labels_train)


In [ ]:
random_forest_pipeline = make_pipeline(
    feature_preprocessor,
    RandomForestClassifier(random_state=42)
)

random_forest_params = {
    'randomforestclassifier__n_estimators': [100, 200],
    'randomforestclassifier__max_depth': [5,10,None]
}

random_forest_search = GridSearchCV(random_forest_pipeline, random_forest_params, scoring='roc_auc', cv=5)
random_forest_search.fit(features_train, labels_train)


In [ ]:
def evaluate_model(model_name, fitted_model):
    predicted_labels = fitted_model.predict(features_test)
    predicted_probabilities = fitted_model.predict_proba(features_test)[:,1]
    
    return {
        'Model': model_name,
        'Accuracy': accuracy_score(labels_test, predicted_labels),
        'Precision': precision_score(labels_test, predicted_labels),
        'Recall': recall_score(labels_test, predicted_labels),
        'F1-Score': f1_score(labels_test, predicted_labels),
        'ROC-AUC': roc_auc_score(labels_test, predicted_probabilities)
    }


In [ ]:
model_results = [
    evaluate_model('Logistic Regression', logistic_search.best_estimator_),
    evaluate_model('KNN', knn_search.best_estimator_),
    evaluate_model('SVM', svm_search.best_estimator_),
    evaluate_model('Decision Tree', decision_tree_search.best_estimator_),
    evaluate_model('Random Forest', random_forest_search.best_estimator_)
]

model_results_df = pd.DataFrame(model_results).sort_values('ROC-AUC', ascending=False)
model_results_df
